In [0]:
storage_account = "stfintechpipeline"

storage_account_key = "YOUR_ACCESS_KEY_HERE"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_account_key
)

RAW = f"abfss://raw@{storage_account}.dfs.core.windows.net"
CLEANED = f"abfss://cleaned@{storage_account}.dfs.core.windows.net"

print("Connection configured.")
print(f"Reading from : {RAW}")
print(f"Writing to   : {CLEANED}")

Connection configured.
Reading from : abfss://raw@stfintechpipeline.dfs.core.windows.net
Writing to   : abfss://cleaned@stfintechpipeline.dfs.core.windows.net


In [0]:
df_transactions = spark.read.format("csv")\
    .option("header", "true")\
     .load(f"{RAW}/transactions_data/transactions_data.csv")
display(df_transactions.limit(5))
df_cards = spark.read.format("csv")\
    .option("header", "true")\
    .load(f"{RAW}/cards_data/cards_data.csv")
display(df_cards.limit(5))
df_users = spark.read.format("csv")\
    .option("header", "true")\
    .load(f"{RAW}/users_data/users_data.csv")
    
print(f"transactions : {df_transactions.count():,} rows")
print(f"cards        : {df_cards.count():,} rows")
print(f"users        : {df_users.count():,} rows")

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,null
7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,null
7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,null
7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,null
7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,null


id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


transactions : 13,305,915 rows
cards        : 6,146 rows
users        : 2,000 rows


In [0]:
import json
#fraud labels
df_fraud_text = spark.read.text(
    "abfss://raw@stfintechpipeline.dfs.core.windows.net/train_fraud_labels.json"
)
df_fraud_raw = df_fraud_text.first()["value"]
fraud_dict = json.loads(df_fraud_raw)["target"]

#convert to Spark dataframe
fraud_list = [(int(k), v) for k, v in fraud_dict.items()]
df_fraud= spark.createDataFrame(fraud_list, ["transaction_id", "is_fraud"])

print(f"fraud labels : {df_fraud.count():,} rows")
df_fraud.show(5)

fraud labels : 8,914,963 rows
+--------------+--------+
|transaction_id|is_fraud|
+--------------+--------+
|      10649266|      No|
|      23410063|      No|
|       9316588|      No|
|      12478022|      No|
|       9558530|      No|
+--------------+--------+
only showing top 5 rows



In [0]:
#since the file loads transposed(each MCC codes is a column) we need to unpivot it into(mcc_code, merchant_category) rows
df_mcc_raw = spark.read.format("json") \
    .option("multiLine", "true") \
    .load(f"{RAW}/mcc_codes.json")
#stack all columns into 2 columns
mcc_cols = df_mcc_raw.columns
stack_expr = f"stack({len(mcc_cols)}, " + \
             ", ".join([f"'{c}', `{c}`" for c in mcc_cols]) + \
             ") as (mcc_code, merchant_category)"
df_mcc = df_mcc_raw.selectExpr(stack_expr) \
    .filter("merchant_category is not null")

print(f"MCC codes : {df_mcc.count():,} rows")
df_mcc.show(5, truncate=False)

MCC codes : 109 rows
+--------+-----------------------------------------------+
|mcc_code|merchant_category                              |
+--------+-----------------------------------------------+
|1711    |Heating, Plumbing, Air Conditioning Contractors|
|3000    |Steelworks                                     |
|3001    |Steel Products Manufacturing                   |
|3005    |Miscellaneous Metal Fabrication                |
|3006    |Miscellaneous Fabricated Metal Products        |
+--------+-----------------------------------------------+
only showing top 5 rows



Clean transactions_data
data cleaning for transactions dataset

In [0]:
from pyspark.sql.functions import (
    col, regexp_replace, to_timestamp, when, lit, trim
)
from pyspark.sql.types import IntegerType, DoubleType, LongType

#cats ID columns from string to integer
df_transactions = df_transactions \
    .withColumn("id", col("id").cast(LongType()))\
    .withColumn("client_id", col("client_id").cast(IntegerType())) \
    .withColumn("card_id", col("card_id").cast(IntegerType())) \
    .withColumn("merchant_id", col("merchant_id").cast(LongType())) \
    .withColumn("mcc", col("mcc").cast(IntegerType()))
  
print("ID columns cast to integer.")
df_transactions.printSchema()


ID columns cast to integer.
root
 |-- id: long (nullable = true)
 |-- date: string (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_id: integer (nullable = true)
 |-- amount: string (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: long (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- errors: string (nullable = true)



In [0]:
#Clean amount column
# Strip $ and commas, cast to double, rename to amount_clean.
# Negative values (refunds) are preserved intentionally.
df_transactions = df_transactions \
    .withColumn("amount_clean",
    regexp_replace(regexp_replace(col("amount"), "\\$", ""), ",", "").cast(DoubleType())) \
    .drop("amount")

#confirm no nulls introduced and negatives preserved
print("amount_clean stats:")
df_transactions.select("amount_clean").summary("min", "max", "count").show()

  

amount_clean stats:
+-------+------------+
|summary|amount_clean|
+-------+------------+
|    min|      -500.0|
|    max|      6820.2|
|  count|    13305915|
+-------+------------+



In [0]:
#parse date column
# Raw format: "2010-01-01 00:01:00" — parse to timestamp,
# preserving the time component for hour-of-day analysis
df_transactions = df_transactions\
    .withColumn("transaction_date",
        to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss")) \
    .drop("date")  


print("transaction_date sample:")
df_transactions.select("transaction_date").show(5)

transaction_date sample:
+-------------------+
|   transaction_date|
+-------------------+
|2010-01-01 00:01:00|
|2010-01-01 00:02:00|
|2010-01-01 00:02:00|
|2010-01-01 00:05:00|
|2010-01-01 00:06:00|
+-------------------+
only showing top 5 rows



In [0]:
#Fill merchant_state nulls
# EDA confirmed: nulls occur where use_chip = "Online Transaction"
# Those fill to "Online". The remaining ~5,788 physical transaction
# nulls fill to "Unknown".
df_transactions = df_transactions \
    .withColumn("merchant_state",
                when(col("merchant_state").isNull()&
                     (col("use_chip")=="Online Transaction"), "Online")
                .when(col("merchant_state").isNull(),"Unknown")
                .otherwise(col("merchant_state")))
#confirm no nulls remain
null_count = df_transactions.filter(col("merchant_state").isNull()).count()

print(f"Remaining merchant_state nulls: {null_count}  (expected 0)")


Remaining merchant_state nulls: 0  (expected 0)


In [0]:
#drop zip- no analytical value added since the null are redundant with merchant_city

df_transactions = df_transactions.drop("zip")
print("zip dropped")
print(f"Remaining columns: {df_transactions.columns}")


zip dropped
Remaining columns: ['id', 'client_id', 'card_id', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'mcc', 'errors', 'amount_clean', 'transaction_date']


In [0]:
#fill errors null. null means the transactions had no error
df_transactions = df_transactions \
    .withColumn("errors",
                when(col("errors").isNull(), "No Error")
                .otherwise(trim(col("errors"))))
#confirm No Error has been added
print("errors value distribution:")
df_transactions.groupBy("errors").count().orderBy("count", ascending=False).show(20)


errors value distribution:
+--------------------+--------+
|              errors|   count|
+--------------------+--------+
|            No Error|13094522|
|Insufficient Balance|  130902|
|             Bad PIN|   32119|
|    Technical Glitch|   26271|
|     Bad Card Number|    7767|
|      Bad Expiration|    6161|
|             Bad CVV|    6106|
|         Bad Zipcode|    1126|
|Bad PIN,Insuffici...|     293|
|Insufficient Bala...|     243|
|Bad Card Number,I...|      71|
|Bad PIN,Technical...|      70|
|Bad CVV,Insuffici...|      57|
|Bad Expiration,In...|      47|
|Bad Card Number,B...|      38|
|Bad Card Number,B...|      33|
|Bad Expiration,Ba...|      32|
|Bad Expiration,Te...|      21|
|Bad Card Number,T...|      15|
|Bad CVV,Technical...|       8|
+--------------------+--------+
only showing top 20 rows



In [0]:
#join fraud labels
df_transactions = df_transactions \
    .join(df_fraud,
          df_transactions.id == df_fraud.transaction_id,
          how="left") \
    .drop("transaction_id") \
    .withColumn("is_fraud",
                when(col("is_fraud").isNull(),"No")
                .otherwise(col("is_fraud")))
    
fraud_counts = df_transactions.groupBy("is_fraud").count()
print("Fraud label distribution:")
fraud_counts.show()


Fraud label distribution:
+--------+--------+
|is_fraud|   count|
+--------+--------+
|      No|13292583|
|     Yes|   13332|
+--------+--------+



In [0]:
# drop all merchant_category duplicates
from pyspark.sql.functions import col

cols_to_keep = [c for c in df_transactions.columns if c != "merchant_category"]
df_transactions = df_transactions.select([col(c) for c in cols_to_keep])

print(df_transactions.columns)

['id', 'client_id', 'card_id', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'mcc', 'errors', 'amount_clean', 'transaction_date', 'is_fraud']


In [0]:
#join mcc codes
df_transactions = df_transactions.alias("txn") \
    .join(df_mcc.alias("mcc_ref"),
          col("txn.mcc").cast("string")== col("mcc_ref.mcc_code"), how="left")\
    .drop("mcc_code")

unmatched = df_transactions.filter(col("merchant_category").isNull()).count()

print(f"Transactions with unmatched MCC code: {unmatched:,}")
df_transactions.select("mcc", "merchant_category").show(5)

Transactions with unmatched MCC code: 0
+----+--------------------+
| mcc|   merchant_category|
+----+--------------------+
|5942|         Book Stores|
|5912|Drug Stores and P...|
|7393|Detective Agencie...|
|5499|Miscellaneous Foo...|
|5942|         Book Stores|
+----+--------------------+
only showing top 5 rows



In [0]:
print("=== Cleaned transactions_data ===")
print(f"Row count : {df_transactions.count():,}")
print(f"Columns   : {len(df_transactions.columns)}")
df_transactions.printSchema()

=== Cleaned transactions_data ===
Row count : 13,305,915
Columns   : 13
root
 |-- id: long (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_id: integer (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: long (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- errors: string (nullable = true)
 |-- amount_clean: double (nullable = true)
 |-- transaction_date: timestamp (nullable = true)
 |-- is_fraud: string (nullable = true)
 |-- merchant_category: string (nullable = true)



Clean cards_data

In [0]:
#drop security columns and zero-variation column

df_cards = df_cards \
    .drop("cvv") \
    .drop("card_number")\
    .drop("card_on_dark_web")
print(f"Remaining columns: {df_cards.columns}")


Remaining columns: ['id', 'client_id', 'card_brand', 'card_type', 'expires', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed']


In [0]:
#strip $ from credit limit,cast to double
#rename to spending limit
#cast id and client_id to integer

df_cards = df_cards \
    .withColumn("spending_limit",
                regexp_replace(regexp_replace(col("credit_limit"), "\\$", ""), ",", "")
                .cast(DoubleType()))\
    .drop("credit_limit")\
    .withColumn("id", col("id").cast(IntegerType()))\
    .withColumn("client_id", col("client_id").cast(IntegerType()))
                
print("spending_limit stats:")
df_cards.select("spending_limit").summary("min", "max", "count").show()


spending_limit stats:
+-------+--------------+
|summary|spending_limit|
+-------+--------------+
|    min|           0.0|
|    max|      151223.0|
|  count|          6146|
+-------+--------------+



In [0]:
from pyspark.sql.functions import upper

# fill null has_chip with unknown
df_cards = df_cards \
    .withColumn("has_chip", 
        when(col("has_chip").isNull(), "Unknown")
        .otherwise(upper(trim(col("has_chip")))))

print("has_chip distribution:")
df_cards.groupBy("has_chip").count().show()

has_chip distribution:
+--------+-----+
|has_chip|count|
+--------+-----+
|     YES| 5500|
|      NO|  646|
+--------+-----+



In [0]:
# ap year_pin_last_changed to dataset end year
# Source data contains values of 2020 which is after the
# dataset end of 2019 — a data quality issue in the raw data
# Any value above 2019 is capped to 2019
df_cards = df_cards.withColumn("year_pin_last_changed",
    when(col("year_pin_last_changed") > 2019, 2019)
    .otherwise(col("year_pin_last_changed")))

print("year_pin_last_changed after cap:")
df_cards.select("year_pin_last_changed").summary("min", "max").show()

year_pin_last_changed after cap:
+-------+---------------------+
|summary|year_pin_last_changed|
+-------+---------------------+
|    min|                 2002|
|    max|                 2019|
+-------+---------------------+



In [0]:
print("=== Cleaned cards_data ===")
print(f"Row count : {df_cards.count():,}")
print(f"Columns   : {len(df_cards.columns)}")
df_cards.printSchema()


=== Cleaned cards_data ===
Row count : 6,146
Columns   : 10
root
 |-- id: integer (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- expires: string (nullable = true)
 |-- has_chip: string (nullable = true)
 |-- num_cards_issued: string (nullable = true)
 |-- acct_open_date: string (nullable = true)
 |-- year_pin_last_changed: string (nullable = true)
 |-- spending_limit: double (nullable = true)



Clean users_data

In [0]:
#strip $ from income and debt columns, cast to double
#drop address-latitude and longitude capture location
#cast if to integer
df_users = df_users \
  .withColumn("yearly_income",
              regexp_replace(regexp_replace(col("yearly_income"), "\\$", ""), ",","")
              .cast(DoubleType()))\
  .withColumn("total_debt",
              regexp_replace(regexp_replace(col("total_debt"), "\\$", ""),",", "")
              .cast(DoubleType())) \
  .withColumn("per_capita_income",
              regexp_replace(regexp_replace(col("per_capita_income"), "\\$", ""),",", "")
               .cast(DoubleType())) \
  .drop("address") \
  .withColumn("id", col("id").cast(IntegerType()))

print("Income/debt stats:")
df_users.select("yearly_income", "total_debt", "per_capita_income") \
    .summary("min", "max", "count").show()

Income/debt stats:
+-------+-------------+----------+-----------------+
|summary|yearly_income|total_debt|per_capita_income|
+-------+-------------+----------+-----------------+
|    min|          1.0|       0.0|              0.0|
|    max|     307018.0|  516263.0|         163145.0|
|  count|         2000|      2000|             2000|
+-------+-------------+----------+-----------------+



In [0]:
print("=== Cleaned users_data ===")
print(f"Row count : {df_users.count():,}")
print(f"Columns   : {len(df_users.columns)}")
df_users.printSchema()

=== Cleaned users_data ===
Row count : 2,000
Columns   : 13
root
 |-- id: integer (nullable = true)
 |-- current_age: string (nullable = true)
 |-- retirement_age: string (nullable = true)
 |-- birth_year: string (nullable = true)
 |-- birth_month: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- per_capita_income: double (nullable = true)
 |-- yearly_income: double (nullable = true)
 |-- total_debt: double (nullable = true)
 |-- credit_score: string (nullable = true)
 |-- num_credit_cards: string (nullable = true)



In [0]:
#fix the remaining string columns in cards and users
#cards
df_cards = df_cards \
    .withColumn("num_cards_issued", col("num_cards_issued").cast(IntegerType()))\
    .withColumn("year_pin_last_changed", col("year_pin_last_changed").cast(IntegerType()))

#users-cast all numeric columns
df_users = df_users \
    .withColumn("current_age",      col("current_age").cast(IntegerType())) \
    .withColumn("retirement_age",   col("retirement_age").cast(IntegerType())) \
    .withColumn("birth_year",       col("birth_year").cast(IntegerType())) \
    .withColumn("birth_month",      col("birth_month").cast(IntegerType())) \
    .withColumn("credit_score",     col("credit_score").cast(IntegerType())) \
    .withColumn("num_credit_cards", col("num_credit_cards").cast(IntegerType())) \
    .withColumn("latitude",         col("latitude").cast(DoubleType())) \
    .withColumn("longitude",        col("longitude").cast(DoubleType()))

print("=== Cards updated schema ===")
df_cards.printSchema()

print("=== Users updated schema ===")
df_users.printSchema()


=== Cards updated schema ===
root
 |-- id: integer (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- expires: string (nullable = true)
 |-- has_chip: string (nullable = true)
 |-- num_cards_issued: integer (nullable = true)
 |-- acct_open_date: string (nullable = true)
 |-- year_pin_last_changed: integer (nullable = true)
 |-- spending_limit: double (nullable = true)

=== Users updated schema ===
root
 |-- id: integer (nullable = true)
 |-- current_age: integer (nullable = true)
 |-- retirement_age: integer (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_month: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- per_capita_income: double (nullable = true)
 |-- yearly_income: double (nullable = true)
 |-- total_debt: double (nullable = true)
 |-- credit_score: integer (n

Write cleaned outputs to ADLS cleaned/container
it will be written as Parquet for faster processing by Synapse

In [0]:
df_transactions.write.mode("overwrite").parquet(f"{CLEANED}/transactions_data/")
print("transactions written")

df_cards.write.mode("overwrite").parquet(f"{CLEANED}/cards_data/")
print("cards written")

df_users.write.mode("overwrite").parquet(f"{CLEANED}/users_data/")
print("users written")

transactions written
cards written
users written


In [0]:
from pyspark.sql import SparkSession

df_check_txn = spark.read.parquet(f"{CLEANED}/transactions_data/")
df_check_cards = spark.read.parquet(f"{CLEANED}/cards_data/")
df_check_users = spark.read.parquet(f"{CLEANED}/users_data/")
print("=== Verification read-back ===")
print(f"transactions : {df_check_txn.count():,} rows, {len(df_check_txn.columns)} columns")
print(f"cards        : {df_check_cards.count():,} rows, {len(df_check_cards.columns)} columns")
print(f"users        : {df_check_users.count():,} rows, {len(df_check_users.columns)} columns")

=== Verification read-back ===
transactions : 13,305,915 rows, 13 columns
cards        : 6,146 rows, 10 columns
users        : 2,000 rows, 13 columns


In [0]:
from pyspark.sql.functions import max as spark_max, min as spark_min
df_transactions.select(
    spark_min("transaction_date").alias("earliest"),
    spark_max("transaction_date").alias("latest")
).show()

+-------------------+-------------------+
|           earliest|             latest|
+-------------------+-------------------+
|2010-01-01 00:01:00|2019-10-31 23:59:00|
+-------------------+-------------------+

